# LTX-Video di Google Colab (Gratis) — untuk NovelOS

Notebook ini menjalankan **LTX-Video** (model video open-weight dari Lightricks) di GPU gratis Google Colab (T4).

**Yang bisa dilakukan:**
- **Text-to-Video:** ketik deskripsi (prompt) → jadi video pendek.
- **Image-to-Video (bonus):** upload gambar panel → jadi video pendek (animasi panel NovelOS).

**Cara pakai:**
1. Buka notebook ini di Google Colab.
2. Pilih **Runtime → Change runtime type → Hardware accelerator = GPU** (pilih T4 kalau ada).
3. Jalankan cell satu per satu dari atas ke bawah (tombol play di kiri tiap cell).

**Catatan penting — baca dulu:**
- **Download model:** otomatis oleh notebook, **pakai internet Google** (bukan kuota HP Tuan). Download pertama besar (puluhan GB), butuh beberapa menit. Sabar di cell load model.
- **Tiap sesi baru, model di-download ulang** (VM Colab itu sementara). Supaya tidak download ulang, ada cell opsional untuk menyimpan model di Google Drive.
- **Internet HP Tuan** hanya dipakai untuk membuka Colab dan mengunduh video hasil (beberapa MB).
- **Kualitas LTX di bawah Veo**, tapi gratis dan tidak pakai kredit. Veo tetap yang terbaik kalau kredit masih ada.
- **LTX-2.5 (yang baru, 22B) TIDAK bisa jalan di Colab gratis** — butuh VRAM 32 GB. Notebook ini memakai LTX-Video (13B) yang memang dirancang muat di T4.


## 1. Cek GPU

Jalankan cell ini dulu. Harus muncul nama GPU (misal **Tesla T4**). Kalau tertulis "TIDAK ADA GPU", ganti runtime ke GPU dulu (Runtime → Change runtime type → GPU).


In [ ]:
import torch
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("TIDAK ADA GPU! Ganti runtime: Runtime -> Change runtime type -> Hardware accelerator = GPU (T4).")


## 2. Install paket

Sekali per sesi. Menginstall diffusers terbaru (dari main) + dependensi. Butuh 1–3 menit.


In [ ]:
# Install diffusers terbaru (dari main) + dependensi. Sekali per sesi.
!pip install -q -U git+https://github.com/huggingface/diffusers
!pip install -q transformers accelerate imageio imageio-ffmpeg
print("Install selesai.")


## 3. (Opsional) Simpan model di Google Drive — supaya tidak download ulang tiap sesi

Jalankan cell di bawah ini **SEBELUM** cell load model (bagian 5). Ini memasang Google Drive dan mengarahkan cache Hugging Face ke Drive, jadi sesi berikutnya model tinggal dibaca dari Drive.

Kalau Tuan skip bagian ini, model tetap jalan — hanya saja tiap sesi baru harus download ulang.


In [ ]:
# OPSIONAL: cache model di Google Drive supaya sesi berikutnya tidak download ulang.
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/ltx_cache'
print("HF_HOME =", os.environ['HF_HOME'])
print("Selesai. Lanjutkan ke cell load model (bagian 5).")


## 4. (Opsional) Login Hugging Face — hanya kalau download gagal (error 401/403)

Model LTX mungkin meminta persetujuan lisensi. Kalau cell load model (bagian 5) gagal dengan error **401/403**, jalankan cell di bawah ini dengan token Hugging Face gratis Tuan (buat di https://huggingface.co/settings/tokens), lalu jalankan ulang cell load model.


In [ ]:
# OPSIONAL: login Hugging Face (hanya kalau download model gagal 401/403).
# Buat token gratis di https://huggingface.co/settings/tokens lalu tempel di bawah.
from huggingface_hub import login
HF_TOKEN = ""  # <-- tempel token di sini, misal "hf_xxxx"
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Login HF berhasil.")
else:
    print("Token kosong — cell ini dilewati. Isi HF_TOKEN kalau download model gagal 401/403.")


## 5. Load pipeline LTX-Video (text-to-video)

Cell ini memuat model + mengoptimasi hemat memori (FP8 + group offloading) supaya muat di GPU T4 gratis (~10 GB VRAM).

**Download pertama bisa beberapa menit (puluhan GB, pakai internet Google).** Tunggu sampai muncul "Pipeline siap."


In [ ]:
# Load pipeline LTX-Video (text-to-video), dioptimasi hemat memori untuk GPU T4 gratis.
# Download pertama bisa beberapa menit (puluhan GB, pakai internet Google).
import torch
from diffusers import LTXPipeline, AutoModel
from diffusers.hooks import apply_group_offloading

# FP8 layerwise casting (hemat VRAM)
transformer = AutoModel.from_pretrained(
    "Lightricks/LTX-Video",
    subfolder="transformer",
    dtype=torch.bfloat16,
)
transformer.enable_layerwise_casting(
    storage_dtype=torch.float8_e4m3fn, compute_dtype=torch.bfloat16
)

pipe = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video", transformer=transformer, dtype=torch.bfloat16
)

# Group offloading (hemat VRAM, muat di T4 ~10GB)
onload_device = torch.device("cuda")
offload_device = torch.device("cpu")
pipe.transformer.enable_group_offload(
    onload_device=onload_device, offload_device=offload_device,
    offload_type="leaf_level", use_stream=True,
)
apply_group_offloading(pipe.text_encoder, onload_device=onload_device, offload_type="block_level", num_blocks_per_group=2)
apply_group_offloading(pipe.vae, onload_device=onload_device, offload_type="leaf_level")

print("Pipeline siap.")


## 6. Generate video dari teks (Text-to-Video)

**Ubah `PROMPT` di bawah sesuai adegan yang Tuan mau**, lalu jalankan cell ini. Video disimpan sebagai `ltx_output.mp4` dan langsung tampil di notebook.

- `NUM_FRAMES`: jumlah frame. Harus 8n+1 (25, 49, 97, 161). 97 frame = ~4 detik @24fps.
- `WIDTH`/`HEIGHT`: harus kelipatan 32. 768×512 = landscape. Untuk vertikal 9:16 (NovelOS), pakai 576×1024.
- `NUM_STEPS`: makin besar makin bagus tapi makin lama. 30 sudah cukup.


In [ ]:
# ===== UBAH PROMPT DI SINI =====
PROMPT = "A young man in an orange hat walks through a quiet village street at golden hour, warm sunlight, cinematic, gentle camera movement"
NEGATIVE_PROMPT = "worst quality, inconsistent motion, blurry, jittery, distorted"
WIDTH = 768
HEIGHT = 512
NUM_FRAMES = 97      # 97 frame = ~4 detik @24fps (harus 8n+1: 25, 49, 97, 161)
NUM_STEPS = 30
SEED = 0
OUT = "ltx_output.mp4"
# ===============================

from diffusers.utils import export_to_video

video = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    width=WIDTH,
    height=HEIGHT,
    num_frames=NUM_FRAMES,
    decode_timestep=0.03,
    decode_noise_scale=0.025,
    num_inference_steps=NUM_STEPS,
    generator=torch.Generator().manual_seed(SEED),
).frames[0]

export_to_video(video, OUT, fps=24)
print("Video tersimpan:", OUT)

# Tampilkan video di notebook
from IPython.display import Video
Video(OUT, embed=True, width=640)


## 7. Download video hasil ke HP

Jalankan cell ini — video akan terunduh ke folder unduhan HP Tuan (beberapa MB, pakai internet HP).


In [ ]:
# Download video hasil ke HP Tuan
from google.colab import files
files.download("ltx_output.mp4")
print("Download dimulai — cek folder unduhan HP Tuan.")


## Bonus: Image-to-Video (animasi gambar panel NovelOS)

Cell di bawah ini menganimasikan sebuah gambar (misal panel manhwa) menjadi video pendek.

**Cara pakai:**
1. Jalankan cell upload (bagian 8) — pilih gambar panel (JPG/PNG).
2. Jalankan cell generate (bagian 9).

Ini memakai pipeline kedua (`LTXConditionPipeline`, model 0.9.8-dev) — jadi akan **download model kedua** (~belasan GB) saat pertama kali dijalankan. Default ukuran **576×1024 (vertikal 9:16)** sesuai format panel NovelOS. Kalau kehabisan memori, ubah ke 512×768.


## 8. Upload gambar panel


In [ ]:
# Upload gambar panel yang mau dianimasikan (JPG/PNG)
from google.colab import files
uploaded = files.upload()
IMG_PATH = list(uploaded.keys())[0]
print("Gambar terupload:", IMG_PATH)


## 9. Generate video dari gambar (Image-to-Video)

Ubah `IMG_PROMPT` sesuai gerakan yang Tuan mau (misal: karakter menoleh, rambut bergerak tertiup angin).


In [ ]:
# Image-to-Video: animasikan gambar panel menjadi video pendek.
# (Memuat pipeline kedua — download pertama bisa beberapa menit.)
import torch
from diffusers import LTXConditionPipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.hooks import apply_group_offloading
from diffusers.utils import export_to_video, load_image, load_video

# --- prompt & setting (ubah sesuai kebutuhan) ---
IMG_PROMPT = "The character in the image slowly looks up, wind moves the hair and clothes gently, cinematic, subtle motion"
IMG_NEGATIVE = "worst quality, inconsistent motion, blurry, jittery, distorted"
IMG_WIDTH = 576    # vertikal 9:16 (NovelOS). Kalau OOM, ubah ke 512.
IMG_HEIGHT = 1024  # vertikal 9:16. Kalau OOM, ubah ke 768.
IMG_FRAMES = 49    # 49 frame = ~2 detik @24fps
IMG_STEPS = 30
IMG_OUT = "ltx_i2v_output.mp4"
# ------------------------------------------------

pipe_i2v = LTXConditionPipeline.from_pretrained(
    "Lightricks/LTX-Video-0.9.8-dev", dtype=torch.bfloat16
)
# Hemat VRAM (sama seperti pipeline text-to-video)
onload_device = torch.device("cuda")
offload_device = torch.device("cpu")
pipe_i2v.transformer.enable_group_offload(
    onload_device=onload_device, offload_device=offload_device,
    offload_type="leaf_level", use_stream=True,
)
apply_group_offloading(pipe_i2v.text_encoder, onload_device=onload_device, offload_type="block_level", num_blocks_per_group=2)
apply_group_offloading(pipe_i2v.vae, onload_device=onload_device, offload_type="leaf_level")
pipe_i2v.vae.enable_tiling()

image = load_image(IMG_PATH).resize((IMG_WIDTH, IMG_HEIGHT))
video_cond = load_video(export_to_video([image]))  # kompres gambar jadi video (model dilatih di video)
condition = LTXVideoCondition(video=video_cond, frame_index=0)

frames = pipe_i2v(
    conditions=[condition],
    prompt=IMG_PROMPT,
    negative_prompt=IMG_NEGATIVE,
    width=IMG_WIDTH,
    height=IMG_HEIGHT,
    num_frames=IMG_FRAMES,
    num_inference_steps=IMG_STEPS,
    decode_timestep=0.05,
    decode_noise_scale=0.025,
    image_cond_noise_scale=0.025,
    generator=torch.Generator().manual_seed(0),
).frames[0]

export_to_video(frames, IMG_OUT, fps=24)
print("Video tersimpan:", IMG_OUT)

from IPython.display import Video
Video(IMG_OUT, embed=True, width=480)


## 10. Download video image-to-video ke HP


In [ ]:
from google.colab import files
files.download("ltx_i2v_output.mp4")
print("Download dimulai — cek folder unduhan HP Tuan.")
